In [6]:
from econml.dml import CausalForestDML
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.tree import DecisionTreeClassifier

### Load data and engineer variable

In [21]:
df = pd.read_csv('../data/software_usage_promotion.csv')

df['Combined Treatment'] = (df['Discount'] *
                            df['Tech Support']).astype('category')
confounders = [
    'Size', 'Employee Count', 'PC Count', 'IT Spend', 'Major Flag',
    'Global Flag', 'Commercial Flag', 'SMC Flag'
]

### Model with CausalForest

In [26]:
def make_cf():
    return CausalForestDML(
        model_y=GradientBoostingRegressor(),
        model_t=DecisionTreeClassifier(max_depth=3),
        discrete_treatment=True,
        n_estimators=100,
        max_depth=10,
        random_state=42
    )

In [27]:
cf_discount = make_cf().fit(Y=df['Revenue'],
       T=df['Discount'],
       X=df[confounders])

cf_tech = make_cf().fit(Y=df['Revenue'], T=df['Tech Support'], X=df[confounders])

cf_combined = make_cf().fit(Y=df['Revenue'], T=df['Combined Treatment'], X=df[confounders])

In [28]:
df['cate_discount'] = cf_discount.effect(df[confounders])
df['cate_tech_support'] = cf_tech.effect(df[confounders])
df['cate_both'] = cf_combined.effect(df[confounders])


In [29]:
df

,Global Flag,Major Flag,SMC Flag,Commercial Flag,IT Spend,Employee Count,PC Count,Size,Tech Support,Discount,Revenue,Combined Treatment,cate_discount,cate_tech_support,cate_both
0,1,0,1,0,45537,26,26,152205,0,1,17688.363000,0,7190.107201,7495.114405,9462.858511
1,0,0,1,1,20842,107,70,159038,0,1,14981.435590,0,7518.531066,7900.942814,10242.973120
2,0,0,0,1,82171,10,7,264935,1,1,32917.138940,1,12926.782373,10147.748368,12637.765833
3,0,0,0,0,30288,40,39,77522,1,1,14773.768550,1,6007.386957,7527.618929,8149.882460
4,0,0,1,0,25930,37,43,91446,1,1,17098.698230,1,4832.062978,7101.963521,8183.235756
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,0,0,1,1,46186,74,48,141579,0,1,13930.128620,0,8122.400114,7651.272712,9741.707479
1996,0,0,1,0,39683,12,13,111848,0,0,4753.072214,0,4949.261027,6819.277791,8544.257533
1997,0,1,0,0,4195,14,17,11924,0,0,2161.745939,0,1023.293220,5274.635512,4916.257254
1998,1,0,0,1,10664,68,47,40037,1,1,17694.820790,1,2091.804664,5724.234631,5776.329695


In [30]:
df.describe()

,Global Flag,Major Flag,SMC Flag,Commercial Flag,IT Spend,Employee Count,PC Count,Size,Tech Support,Discount,Revenue,cate_discount,cate_tech_support,cate_both
count,2000.000000,2000.0000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000
mean,0.202000,0.1950,0.504500,0.691000,28272.703000,61.115000,57.345500,113159.120000,0.503000,0.510500,15397.917422,5433.324485,6758.105835,7924.313022
std,0.401593,0.3963,0.500105,0.462197,28207.138885,53.449707,52.861123,100987.600092,0.500116,0.500015,11290.944451,4648.452379,1216.389386,2905.237455
min,0.000000,0.0000,0.000000,0.000000,1161.000000,10.000000,6.000000,10101.000000,0.000000,0.000000,-616.572451,-354.962442,4850.491899,3681.432223
25%,0.000000,0.0000,0.000000,0.000000,8914.000000,24.000000,22.000000,39282.000000,0.000000,0.000000,7545.052008,1626.249922,5696.380856,5308.872987
50%,0.000000,0.0000,1.000000,1.000000,19210.500000,44.000000,41.000000,81378.000000,1.000000,1.000000,12582.446440,4587.119452,6589.175979,7783.148574
75%,0.000000,0.0000,1.000000,1.000000,37991.500000,79.000000,74.000000,155635.000000,1.000000,1.000000,19662.979475,7961.101743,7596.104630,9810.128221
max,1.000000,1.0000,1.000000,1.000000,259808.000000,535.000000,407.000000,766485.000000,1.000000,1.000000,86006.924450,21160.666596,11334.062919,18400.823534


In [32]:
df[df.cate_discount<0].shape

In [ ]:
if discount
    move_to_both = check cate_both - cate_discount
    move_to_tech = check_cate_tech - cate_discount
    move_to_nothing = -cate_discount

if tech
    move_to_both = cate-both - cate_tech
    move_to_discount = cate_discount - cate_tech
    move_to_nothing = -cate_tech

if both
    move_to_tech = cate_tech - cate_both
    move_to_discount = cate_discount - cate_both
    move_to_nothing = -cate_both

In [ ]:
def row_optimal_strategy(row):
    discount = row['Discount']
    tech = row['Tech Support']

    match (discount, tech):
        case (0, 0):  # None
            lifts = {
                'none': 0,
                'discount': row['cate_discount'],
                'tech_support': row['cate_tech_support'],
                'both': row['cate_both']
            }
        case (1, 0): # Discount
            lifts = {
                'none': -row['cate_discount'],
                'discount': 0,
                'tech_support': row['cate_tech_support'] - row['cate_discount'],
                'both': row['cate_both'] - row['cate_discount']
            }
        case (0, 1): # Tech
            lifts = {
                'none': -row['cate_tech_support'],
                'discount': row['cate_tech_discount'] - row['cate_tech_support'],
                'tech_support': 0,
                'both': row['cate_both'] - row['cate_tech']
            }
        case (1, 1): # Both
            lifts = {
                'none': -row['cate_both'],
                'discount': row['cate_discount'] - row['cate_both'],
                'tech_support': row['cate_tech_support'] - row['cate_both'],
                'both': 0
            }

        case _:  # fallback
            raise ValueError("Unexpected treatment combination")

    best_action = max(lifts, key=lifts.get)
    best_lift = lifts[best_action]

    return pd.Series({'recommended_action': best_action, 'incremental_lift': best_lift})


In [35]:
df[['recommended_action', 'incremental_lift']] = df.apply(row_optimal_strategy, axis=1)

KeyError: 'cate_tech_discount'